In [1]:
import tensorflow as tf
import numpy as np
import os
import logging
import string
import random
import yaml
from datetime import datetime
from tqdm.notebook import tqdm

# 可视化库
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

from dimenet.model.dimenet import DimeNet
from dimenet.model.dimenet_pp import DimeNetPP
from dimenet.model.activations import swish
from dimenet.training.trainer import Trainer
from dimenet.training.metrics import Metrics
from dimenet.training.data_container import DataContainer
from dimenet.training.data_provider import DataProvider

# 设置matplotlib中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

/opt/anaconda3/envs/lewis_acid_env/lib/python3.9/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
# Set up logger
logger = logging.getLogger()
logger.handlers = []
ch = logging.StreamHandler()
formatter = logging.Formatter(
        fmt='%(asctime)s (%(levelname)s): %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S')
ch.setFormatter(formatter)
logger.addHandler(ch)
logger.setLevel('INFO')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'
tf.get_logger().setLevel('WARN')
tf.autograph.set_verbosity(2)

### Load config

In [3]:
# 使用大数据集配置文件
with open('config_pp_minimal.yaml', 'r') as c:
    config = yaml.safe_load(c)
    
print("Configuration loaded:")
print(f"Model: {config['model_name']}")
print(f"Targets: {config['targets']}")
print(f"Dataset: {config['dataset']}")
print(f"Minimal output: {config.get('minimal_output', False)}")
print(f"Freeze features: {config.get('freeze_features', False)}")
print(f"Training samples: {config['num_train']}")
print(f"Validation samples: {config['num_valid']}")
print(f"Test samples: {48986 - config['num_train'] - config['num_valid']}")

Configuration loaded:
Model: dimenet++
Targets: ['fia_gas-DSDBLYP']
Dataset: data/fia49k_processed.npz
Minimal output: True
Freeze features: True
Training samples: 1600
Validation samples: 300
Test samples: 47086


In [4]:
model_name = config['model_name']

if model_name == "dimenet":
    num_bilinear = config['num_bilinear']
elif model_name == "dimenet++":
    out_emb_size = config['out_emb_size']
    int_emb_size = config['int_emb_size']
    basis_emb_size = config['basis_emb_size']
    extensive = config['extensive']
else:
    raise ValueError(f"Unknown model name: '{model_name}'")
    
emb_size = config['emb_size']
num_blocks = config['num_blocks']

num_spherical = config['num_spherical']
num_radial = config['num_radial']
output_init = config['output_init']

cutoff = config['cutoff']
envelope_exponent = config['envelope_exponent']

num_before_skip = config['num_before_skip']
num_after_skip = config['num_after_skip']
num_dense_output = config['num_dense_output']

num_train = config['num_train']
num_valid = config['num_valid']
data_seed = config['data_seed']
dataset_path = config['dataset']

batch_size = config['batch_size']

# 新增
minimal_output = config.get('minimal_output', False)
freeze_features = config.get('freeze_features', False)
pretrained_weights_path = config.get('pretrained_weights_path', None)

#####################################################################
# Change this if you want to predict a different target, e.g. to ['U0']
# (but don't forget to change output_init as well)
targets = config['targets']
#####################################################################

### Load dataset

In [5]:
print(f"Loading dataset from: {dataset_path}")
data_container = DataContainer(dataset_path, cutoff=cutoff, target_keys=targets)

# Initialize DataProvider (splits dataset into training, validation and test set based on data_seed)
data_provider = DataProvider(data_container, num_train, num_valid, batch_size,
                             seed=data_seed, randomized=True)

print(f"Dataset loaded successfully!")
print(f"Training samples: {data_provider.nsamples['train']}")
print(f"Test samples: {data_provider.nsamples['test']}")

print(f"\n=== 数据划分确认 ===")
print(f"✅ 我们正在预测测试集，包含 {data_provider.nsamples['test']} 个样本")
print(f"✅ 这些样本在训练过程中从未被模型见过")
print(f"✅ 数据划分使用随机种子: {data_seed}")

# Initialize datasets
dataset = data_provider.get_dataset('test').prefetch(tf.data.experimental.AUTOTUNE)
dataset_iter = iter(dataset)
print("Test dataset iterator created successfully!")

2025-10-15 21:57:00.877767: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Loading dataset from: data/fia49k_processed.npz
数据加载完成，使用原始目标值
Dataset loaded successfully!
Training samples: 1600
Test samples: 315

=== 数据划分确认 ===
✅ 我们正在预测测试集，包含 315 个样本
✅ 这些样本在训练过程中从未被模型见过
✅ 数据划分使用随机种子: 42
Metal device set to: Apple M1

systemMemory: 8.00 GB
maxCacheSize: 2.67 GB

Test dataset iterator created successfully!


### Initialize model

In [6]:
if model_name == "dimenet":
    model = DimeNet(
            emb_size=emb_size, num_blocks=num_blocks, num_bilinear=num_bilinear,
            num_spherical=num_spherical, num_radial=num_radial,
            cutoff=cutoff, envelope_exponent=envelope_exponent,
            num_before_skip=num_before_skip, num_after_skip=num_after_skip,
            num_dense_output=num_dense_output, num_targets=len(targets),
            activation=swish, output_init=output_init)
elif model_name == "dimenet++":
    print("Initializing DimeNet++ model with parameters:")
    print(f"  emb_size: {emb_size}")
    print(f"  out_emb_size: {out_emb_size}")
    print(f"  int_emb_size: {int_emb_size}")
    print(f"  basis_emb_size: {basis_emb_size}")
    print(f"  num_blocks: {num_blocks}")
    print(f"  minimal_output: {minimal_output}")
    print(f"  freeze_features: {freeze_features}")
    print(f"  pretrained_weights_path: {pretrained_weights_path}")
    
    model = DimeNetPP(
            emb_size=emb_size, out_emb_size=out_emb_size,
            int_emb_size=int_emb_size, basis_emb_size=basis_emb_size,
            num_blocks=num_blocks, num_spherical=num_spherical, num_radial=num_radial,
            cutoff=cutoff, envelope_exponent=envelope_exponent,
            num_before_skip=num_before_skip, num_after_skip=num_after_skip,
            num_dense_output=num_dense_output, num_targets=len(targets),
            activation=swish, extensive=extensive, output_init=output_init,
            minimal_output=minimal_output,
            freeze_features=freeze_features,
            pretrained_weights_path=pretrained_weights_path)
else:
    raise ValueError(f"Unknown model name: '{model_name}'")

Initializing DimeNet++ model with parameters:
  emb_size: 128
  out_emb_size: 256
  int_emb_size: 64
  basis_emb_size: 8
  num_blocks: 4
  minimal_output: True
  freeze_features: True
  pretrained_weights_path: pretrained/dimenet_pp/U/ckpt
冻结特征层完成，输出层保持可训练


### Initialize trainer

In [7]:
trainer = Trainer(model, early_stopping_patience=0)

### Load weights from model at best step

In [8]:
#####################################################################
# Load the trained model from your own training run
# 使用大数据集训练保存的权重
# 注意：训练完成后，请将下面的目录路径更新为实际的训练结果目录
# 训练结果目录格式：YYYYMMDD_HHMMSS_随机ID_fia49k_all_processed.npz_fia_gas-DSDBLYP_Large FIA dataset minimal output

# 使用最佳权重tar包
import tarfile
import tempfile

# 最佳权重文件路径
best_weights_tar = "20251009_214042_xo8fhxiW_fia49k_processed.npz_fia_gas-DSDBLYP_FIA minimal output.tar"
print(f"正在加载最佳权重: {best_weights_tar}")

if os.path.exists(best_weights_tar):
    try:
        # 解压tar文件并查找checkpoint
        with tempfile.TemporaryDirectory() as temp_dir:
            with tarfile.open(best_weights_tar, 'r') as tar:
                tar.extractall(temp_dir)
            
            # 查找best目录下的checkpoint
            best_checkpoint = os.path.join(temp_dir, 'best', 'checkpoint')
            
            if os.path.exists(best_checkpoint):
                # 使用TensorFlow checkpoint加载
                checkpoint = tf.train.Checkpoint(model=model)
                checkpoint.restore(best_checkpoint)
                print(f"✅ 成功加载最佳权重: {best_checkpoint}")
                best_ckpt_file = best_checkpoint  # 为了兼容后续代码
            else:
                print(f"❌ 未找到checkpoint文件: {best_checkpoint}")
                print("将使用随机初始化权重")
                best_ckpt_file = None
    except Exception as e:
        print(f"❌ 加载权重失败: {e}")
        print("将使用随机初始化权重")
        best_ckpt_file = None
else:
    print(f"❌ 未找到权重文件: {best_weights_tar}")
    print("将使用随机初始化权重")
    best_ckpt_file = None
#####################################################################
if best_ckpt_file:
    print(f"权重加载完成: {best_ckpt_file}")
else:
    print("使用随机初始化权重进行预测")

正在加载最佳权重: 20251009_214042_xo8fhxiW_fia49k_processed.npz_fia_gas-DSDBLYP_FIA minimal output.tar
❌ 加载权重失败: Unable to open table file /var/folders/qz/m_320ms557l2j6qhqq83m1c40000gn/T/tmpq3xn1ji2/best/checkpoint: DATA_LOSS: not an sstable (bad magic number): perhaps your file is in a different file format and you need to use a different restore operator?
将使用随机初始化权重
使用随机初始化权重进行预测


2025-10-15 21:57:08.866143: W tensorflow/core/util/tensor_slice_reader.cc:97] Could not open /var/folders/qz/m_320ms557l2j6qhqq83m1c40000gn/T/tmpq3xn1ji2/best/checkpoint: DATA_LOSS: not an sstable (bad magic number): perhaps your file is in a different file format and you need to use a different restore operator?


In [9]:
### 验证模型权重加载
# 检查模型是否已正确加载权重
print("验证模型权重加载状态...")
print(f"模型可训练参数数量: {len(model.trainable_variables)}")

# 显示一些关键层的权重信息
for i, var in enumerate(model.trainable_variables[:5]):  # 只显示前5个变量
    print(f"参数 {i+1}: {var.name}, 形状: {var.shape}, 数据类型: {var.dtype}")

print("权重验证完成!")


验证模型权重加载状态...
模型可训练参数数量: 0
权重验证完成!


### Prediction loop

In [10]:
# Initialize aggregates
metrics = Metrics('val', targets)
preds_total = np.zeros([data_provider.nsamples['test'], len(targets)], dtype=np.float32)
targets_total = np.zeros([data_provider.nsamples['test'], len(targets)], dtype=np.float32)  # 保存真实值

In [11]:
steps_per_epoch = int(np.ceil(data_provider.nsamples['test'] / batch_size))

for step in tqdm(range(steps_per_epoch)):
    # 获取预测值和真实值
    inputs, targets_batch = next(dataset_iter)
    preds = model(inputs, training=False)
    
    # 更新指标
    mae = tf.reduce_mean(tf.abs(targets_batch - preds), axis=0)
    mean_mae = tf.reduce_mean(mae)
    loss = mean_mae
    nsamples = tf.shape(preds)[0]
    metrics.update_state(loss, mean_mae, mae, nsamples)
    
    # 保存预测值和真实值
    batch_start = step * batch_size
    batch_end = min((step + 1) * batch_size, data_provider.nsamples['test'])
    preds_total[batch_start:batch_end] = preds.numpy()
    targets_total[batch_start:batch_end] = targets_batch.numpy()

  0%|          | 0/20 [00:00<?, ?it/s]

/opt/anaconda3/envs/lewis_acid_env/lib/python3.9/site-packages/keras/initializers/initializers.py:120: UserWarning: The initializer Orthogonal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
print(f"\n=== 测试集预测结果 ===")
print(f"✅ 测试集样本数: {data_provider.nsamples['test']}")
print(f"✅ 目标属性: {','.join(targets)}")
print(f"✅ 测试集MAE: {metrics.mean_mae:.6f}")
print(f"✅ 测试集logMAE: {metrics.mean_log_mae:.6f}")

# 显示一些预测样本
print(f"\n=== 测试集预测样本 (前10个) ===")
for i in range(min(10, len(preds_total))):
    print(f"测试样本 {i+1}: {preds_total[i][0]:.6f}")

# 保存预测结果
output_file = f"test_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.npz"
np.savez(output_file, 
         predictions=preds_total, 
         targets=targets,
         mae=metrics.mean_mae,
         log_mae=metrics.mean_log_mae,
         test_samples=data_provider.nsamples['test'],
         data_seed=data_seed)
print(f"\n✅ 测试集预测结果已保存到: {output_file}")
print(f"✅ 这是真正的测试集预测，模型在训练时从未见过这些样本")

In [ ]:
### 可视化预测结果
print("开始生成可视化图表...")

# 提取预测值和真实值
y_true = targets_total[:, 0]  # 真实值
y_pred = preds_total[:, 0]    # 预测值
errors = y_pred - y_true      # 预测误差

# 计算额外的评估指标
mae_sklearn = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(np.mean(errors**2))

print(f"详细评估指标:")
print(f"MAE: {mae_sklearn:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²: {r2:.6f}")


In [ ]:
# 创建图表
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('测试集预测结果可视化', fontsize=16, fontweight='bold')

# 1. 预测值 vs 真实值散点图
ax1 = axes[0, 0]
ax1.scatter(y_true, y_pred, alpha=0.6, s=20)
min_val = min(y_true.min(), y_pred.min())
max_val = max(y_true.max(), y_pred.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='完美预测线')
ax1.set_xlabel('真实值')
ax1.set_ylabel('预测值')
ax1.set_title(f'预测值 vs 真实值\nR² = {r2:.4f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 预测误差分布直方图
ax2 = axes[0, 1]
ax2.hist(errors, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='零误差线')
ax2.set_xlabel('预测误差 (预测值 - 真实值)')
ax2.set_ylabel('频次')
ax2.set_title(f'预测误差分布\nMAE = {mae_sklearn:.4f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 残差图
ax3 = axes[1, 0]
ax3.scatter(y_pred, errors, alpha=0.6, s=20)
ax3.axhline(0, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('预测值')
ax3.set_ylabel('残差 (预测值 - 真实值)')
ax3.set_title('残差图')
ax3.grid(True, alpha=0.3)

# 4. 预测样本对比 (前20个样本)
ax4 = axes[1, 1]
sample_indices = np.arange(min(20, len(y_true)))
x_pos = np.arange(len(sample_indices))
width = 0.35

ax4.bar(x_pos - width/2, y_true[sample_indices], width, label='真实值', alpha=0.8)
ax4.bar(x_pos + width/2, y_pred[sample_indices], width, label='预测值', alpha=0.8)
ax4.set_xlabel('样本索引')
ax4.set_ylabel('目标值')
ax4.set_title('前20个测试样本对比')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("可视化图表生成完成!")


In [ ]:
### 详细预测分析
print("=== 详细预测分析 ===")

# 创建DataFrame用于分析
df_results = pd.DataFrame({
    '真实值': y_true,
    '预测值': y_pred,
    '误差': errors,
    '绝对误差': np.abs(errors),
    '相对误差': np.abs(errors) / (np.abs(y_true) + 1e-8) * 100
})

print(f"预测统计信息:")
print(f"真实值范围: [{y_true.min():.4f}, {y_true.max():.4f}]")
print(f"预测值范围: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
print(f"误差范围: [{errors.min():.4f}, {errors.max():.4f}]")
print(f"平均绝对误差: {df_results['绝对误差'].mean():.4f}")
print(f"误差标准差: {errors.std():.4f}")
print(f"最大绝对误差: {df_results['绝对误差'].max():.4f}")
print(f"平均相对误差: {df_results['相对误差'].mean():.2f}%")

# 显示预测最好的和最差的样本
print(f"\n预测最好的10个样本 (绝对误差最小):")
best_samples = df_results.nsmallest(10, '绝对误差')
for idx, row in best_samples.iterrows():
    print(f"样本 {idx}: 真实值={row['真实值']:.4f}, 预测值={row['预测值']:.4f}, 误差={row['误差']:.4f}")

print(f"\n预测最差的50个样本 (绝对误差最大):")
worst_samples = df_results.nlargest(50, '绝对误差')
for idx, row in worst_samples.iterrows():
    print(f"样本 {idx}: 真实值={row['真实值']:.4f}, 预测值={row['预测值']:.4f}, 误差={row['误差']:.4f}")

# 保存详细结果
detailed_results_file = f"detailed_test_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_results.to_csv(detailed_results_file, index=True)
print(f"\n详细预测结果已保存到: {detailed_results_file}")
